In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

products = []
num_pages = 7

for i in range(1, num_pages + 1):

    if i == 1:
        url = "https://www.laptop.lk/?s=&product_cat=laptops&post_type=product"
    else:
        url = f"https://www.laptop.lk/page/{i}/?s&product_cat=laptops&post_type=product"

    print(f"Scraping page: {i}")

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    items = soup.find_all("li", class_="product")

    for item in items:
        try:

            title = item.find("h2", class_="woocommerce-loop-product__title")
            title_text = title.text.strip() if title else "N/A"

            link = item.find("a", class_="woocommerce-LoopProduct-link")
            product_url = link["href"] if link else "N/A"

            image = item.find("img")
            image_url = image["src"] if image else "N/A"

            price = item.find("span", class_="woocommerce-Price-amount")
            price_text = price.text.strip() if price else "N/A"

            description = "N/A"

            if product_url != "N/A":
                product_page = requests.get(product_url, headers=headers)
                product_soup = BeautifulSoup(product_page.text, "html.parser")

                desc_div = product_soup.find(
                    "div",
                    class_="woocommerce-product-details__short-description"
                )

                if desc_div:
                    description = desc_div.get_text(separator=" | ").strip()

                time.sleep(1)  

            products.append({
                "Product Name": title_text,
                "Current Price": price_text,
                "Description": description,
                "Product URL": product_url,
                "Image URL": image_url
            })

            print("Saved:", title_text)

        except Exception as e:
            print("Error scraping product:", e)

    time.sleep(2)

df = pd.DataFrame(products)

for col in ["Current Price"]:
    df[col] = (
        df[col]
        .str.replace("රු", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

df["Current Price"] = pd.to_numeric(df["Current Price"], errors="coerce")

df.to_csv("laptop_data.csv", index=False)


Scraping page: 1
Saved: MSI Crosshair 16 HX AI D2XWGKG – Ultra 9
Saved: Asus Zenbook DUO UX8406 – Ultra 9
Saved: Lenovo Legion 5 15AKP10 Gaming – Ryzen AI 7
Saved: MSI Summit 16 AI Evo 2 in 1 A2HMTG – Ultra 7
Saved: Lenovo Legion 5 15AKP10 Gaming – Ryzen AI 7
Saved: MSI Cyborg 15 B2RWFKG – Core 7
Saved: MSI Katana 15 B14WEK Intel – i7
Saved: MSI Cyborg 15 B2RWEKG Gaming – Core 7
Saved: Dell Latitude 7430 – i5
Saved: Asus Vivobook S 15 S5507 -Snapdragon X Plus X1P
Saved: MSI Cyborg Gaming 15 AI A1VEK – Ultra 7
Saved: Lenovo Yoga 7 2-in-1 14AKP10 – Ryzen AI 7
Saved: Lenovo IdeaPad Pro 5 Creator 16AHP9 – Ryzen 7 (16GB)
Saved: Dell 5525 G15 Gaming – Ryzen 5
Saved: Lenovo LOQ Gaming 15AHP10 – Ryzen 7
Saved: HP EliteBook 8 G1i 14 – Ultra 7
Saved: Lenovo Yoga 7 2-in-1 14AKP10 – Ryzen AI 5
Saved: Lenovo LOQ Gaming 15AHP10 –  Ryzen 5
Saved: HP ProBook 4 G1i 14 – Ultra 7
Saved: HP EliteBook 6 G1i 14 – Ultra 5
Saved: Asus Vivobook S14 S3407QA – Snapdragon
Saved: MSI Cyborg 15 A2RUDX Gaming – Core

In [2]:
df.head()

,Product Name,Current Price,Description,Product URL,Image URL
0,MSI Crosshair 16 HX AI D2XWGKG – Ultra 9,845000.00,| – Intel Core Ultra 9 AI 275HX Processor | \n...,https://www.laptop.lk/product/msi-crosshair-16...,https://www.laptop.lk/wp-content/uploads/2025/...
1,Asus Zenbook DUO UX8406 – Ultra 9,779000.00,| – Intel Core Ultra 9 285H Processor | \n– 1T...,https://www.laptop.lk/product/asus-zenbook-duo...,https://www.laptop.lk/wp-content/uploads/2024/...
2,Lenovo Legion 5 15AKP10 Gaming – Ryzen AI 7,650000.00,| – AMD Ryzen AI 7 350 Processor | \n– Integra...,https://www.laptop.lk/product/lenovo-legion-5-...,https://www.laptop.lk/wp-content/uploads/2025/...
3,MSI Summit 16 AI Evo 2 in 1 A2HMTG – Ultra 7,621000.00,| – Intel Core Ultra 7 processor | \n– 1TB Gen...,https://www.laptop.lk/product/msi-summit-16-ai...,https://www.laptop.lk/wp-content/uploads/2025/...
4,Lenovo Legion 5 15AKP10 Gaming – Ryzen AI 7,615000.00,| – AMD Ryzen AI 7 350 Processor | \n– Integra...,https://www.laptop.lk/product/lenovo-legion-5-...,https://www.laptop.lk/wp-content/uploads/2025/...
